# Chapter 8: Baire Spaces and Dimension Theory

Source orientation: printed pages 292-316; sections 48-50. This notebook is original course material. It uses the source only to orient the order of topics: Baire spaces, the Baire category theorem, nowhere-differentiable functions, and covering dimension.

## Chapter Goal

The chapter's main message is that "large" and "small" in topology are not just cardinality or measure. A countable union of closed sets with empty interior is small in a Baire space; a countable intersection of dense open sets remains dense; a dense family of very rough continuous functions exists in the complete function space; and a space has dimension at most `m` when every open cover can be refined so that no point is forced into more than `m + 1` sets.

## Visualization Storyboard

chapter goal: Make Baire category and covering dimension inspectable: nested choices avoid countably many nowhere-dense obstacles, dense `G_delta` sets behave as residual large sets, rough functions are organized by dense open difference-quotient conditions, and dimension is controlled by refinement order.

source span read: `Topology.pdf`, printed pp. 292-316 / sections 48-50, using the course source map and `pdftotext` because the PDF is imposed. I also inspected `Topology/AGENTS.md`, `scripts/topology_inventory.py`, the chapter index, the existing canonical notebook, and the Chapter 8 artifact subtree.

concept inventory: Baire space closed-set definition; dense-open equivalence; Baire category theorem for compact Hausdorff and complete metric spaces; nested closed-set proof move; open subspaces and `G_delta` Baire examples; pointwise limits of continuous functions; the `U_n` difference-quotient sets in `C([0,1], R)`; dense sawtooth approximations; covering order; open refinements; `dim X <= m`; closed-subspace and finite-union dimension rules; general position; the `2m + 1` embedding mechanism; compact locally Euclidean atlas consequences.

library routing table:

| Concept | Representation | Library | Why this library |
| --- | --- | --- | --- |
| Baire proof state | Directed dependency graph | NetworkX + Matplotlib | The proof is a chain of choices and invariants; graph layout exposes dependencies. |
| Category game and dense `G_delta` | Interval diagrams and finite witness table | NumPy + Matplotlib | The theorem is infinite, but finite rounds show what is being preserved. |
| Nowhere-differentiable intuition | Sawtooth approximants and secant diagnostics | NumPy + Matplotlib + Pandas | Difference quotients are numerical objects that can be sampled and checked. |
| Covering dimension | Refinement-order diagrams and sample overlap counts | NumPy + Matplotlib + Pandas | Dimension is an overlap invariant of covers. |
| General position and embedding | 3D finite graph plus determinant checks | NumPy + Matplotlib 3D | The source proof uses affine independence; determinants are the finite check. |
| Applied lab | Standalone interactive slider | Plotly | Rounds and cover scale are parameters learners should vary. |

visual sequence: proof-state graph, finite Baire category game, dense `G_delta` sieve, rough sawtooth approximation, difference-quotient diagnostics, cover-refinement order checks, general-position embedding sketch, locally Euclidean atlas/refinement witness, and an interactive lab.

computational checks: artifact existence and nonblank PNG statistics; nested interval lengths and avoidance flags; finite dense `G_delta` witnesses; uniform sawtooth error and slope thresholds; sampled cover-order bounds; affine determinant nonzero checks; JSON and CSV summaries.

proof-visualization strategy: expose the Baire theorem as a state machine, the rough-function theorem as a dense-open intersection of finite quotient tests, and dimension as an overlap-count invariant preserved by refinements.

risk notes: finite samples never prove the infinite theorems. The notebook labels them as computational shadows and keeps theorem statements separate from numerical diagnostics.

## Computational Translation Guide

The source chapter is proof-heavy, so the notebook uses three kinds of computational shadows.

First, a countable family of closed nowhere-dense sets is modeled by a finite list of obstacles. A Baire proof does not remove the obstacles; it repeatedly chooses smaller open sets whose closures avoid the next obstacle. The invariant is the nested chain and the fact that a limit point remains in every chosen closure.

Second, a dense `G_delta` set is modeled by a finite intersection of dense open sets. The finite model can only show the inspection habit: every test interval should still contain a witness after finitely many removals. The theorem says this persists countably in Baire spaces.

Third, covering dimension becomes an overlap-count problem. A cover has order at most `m + 1` when no point lies in more than `m + 1` of its sets. A refinement makes sets smaller while preserving coverage. In compact metric examples, mesh scale plays the role of choosing refinements subordinate to a given open cover.

In [ ]:
from __future__ import annotations

from fractions import Fraction
from pathlib import Path
import itertools
import json
import math
import sys

import matplotlib.pyplot as plt
from matplotlib.patches import Circle, FancyArrowPatch, Polygon, Rectangle
from mpl_toolkits.mplot3d.art3d import Line3DCollection
import networkx as nx
import numpy as np
import pandas as pd
import plotly.graph_objects as go
from plotly.subplots import make_subplots

HERE = Path.cwd().resolve()
BOOK_ROOT = HERE if (HERE / "AGENTS.md").exists() else HERE.parent
while not (BOOK_ROOT / "AGENTS.md").exists() and BOOK_ROOT != BOOK_ROOT.parent:
    BOOK_ROOT = BOOK_ROOT.parent
if str(BOOK_ROOT) not in sys.path:
    sys.path.insert(0, str(BOOK_ROOT))

from utils.artifacts import (
    ARTIFACT_ROOT,
    assert_artifacts,
    book_relative,
    display_artifact,
    ensure_artifact_root,
    image_stats,
    save_html,
    save_json,
    save_table,
)
from utils.validation import assert_png_nonblank

CHAPTER_ARTIFACT = ensure_artifact_root(ARTIFACT_ROOT / "chapter-08")
FIGURES = CHAPTER_ARTIFACT / "figures"
HTML = CHAPTER_ARTIFACT / "html"
CHECKS = CHAPTER_ARTIFACT / "checks"
TABLES = CHAPTER_ARTIFACT / "tables"

plt.rcParams.update({"figure.dpi": 120, "savefig.dpi": 170, "font.size": 10})

def save_figure(fig, filename: str) -> Path:
    path = FIGURES / filename
    fig.savefig(path, bbox_inches="tight", facecolor="white")
    plt.close(fig)
    return path

def max_interval_order(intervals, samples=2001):
    grid = np.linspace(0.0, 1.0, samples)
    counts = np.zeros_like(grid, dtype=int)
    for left, right in intervals:
        counts += ((grid > left) & (grid < right)).astype(int)
    return int(counts.max()), grid, counts

def triangular_wave(u):
    frac = u - np.floor(u)
    return 1.0 - np.abs(2.0 * frac - 1.0)

def base_curve(x):
    return 0.16 * np.sin(2 * np.pi * x) + 0.04 * np.cos(6 * np.pi * x)

def rough_series(x, levels=7, amplitude=0.055, ratio=0.62):
    y = base_curve(x).copy()
    for k in range(levels):
        y += amplitude * (ratio ** k) * (triangular_wave((2 ** (k + 1)) * x + 0.17 * k) - 0.5)
    return y

def steep_approximant(x, n, epsilon=0.11):
    base_derivative_bound = 0.16 * 2 * np.pi + 0.04 * 6 * np.pi
    target_slope = n + base_derivative_bound + 1.0
    amp = epsilon / (2 * (n + 1))
    freq = int(math.ceil(target_slope / (2 * amp)))
    return base_curve(x) + amp * (triangular_wave(freq * x + 0.13 * n) - 0.5), amp, freq, target_slope

def finite_secant_score(n, grid):
    _, amp, freq, target = steep_approximant(grid, n)
    def f(z):
        z = np.asarray(z)
        return base_curve(z) + amp * (triangular_wave(freq * z + 0.13 * n) - 0.5)
    hs = np.array([1 / (8 * freq), 1 / (16 * freq), 1 / (32 * freq)])
    values = []
    for x0 in grid:
        local = []
        for h in hs:
            if x0 + h <= 1:
                local.append(abs((f(x0 + h) - f(x0)) / h))
            if x0 - h >= 0:
                local.append(abs((f(x0 - h) - f(x0)) / h))
        values.append(max(local) if local else np.nan)
    values = np.asarray(values, dtype=float)
    epsilon = 0.11
    return {
        "n": int(n),
        "epsilon": float(epsilon),
        "amplitude": float(amp),
        "frequency": int(freq),
        "target_saw_slope": float(target),
        "min_sampled_witness": float(np.nanmin(values)),
        "median_sampled_witness": float(np.nanmedian(values)),
        "p10_sampled_witness": float(np.nanpercentile(values, 10)),
        "p90_sampled_witness": float(np.nanpercentile(values, 90)),
        "uniform_error_bound": float(amp),
    }

def determinant_for_points(points):
    matrix = np.c_[points, np.ones(len(points))]
    return float(np.linalg.det(matrix))

print(f"Book root: {BOOK_ROOT}")
print(f"Chapter artifact root: {book_relative(CHAPTER_ARTIFACT)}")

## Baire Category as a Finite Game

The Baire condition has two equivalent faces.

Closed-set face: a countable union of closed sets with empty interior cannot contain a nonempty open set.

Open-set face: a countable intersection of dense open sets is dense.

The finite game below models the proof for complete metric spaces. Each round presents a closed nowhere-dense obstacle. The player chooses a smaller open interval whose closure avoids that obstacle. Shrinking diameters and completeness are the reason the nested closed intervals keep a point in their intersection. In the actual theorem, the obstacles are closed sets with empty interior; the finite picture uses singleton obstacles because they are easy to see and already capture the category move.

In [ ]:
proof_graph = nx.DiGraph()
proof_nodes = {
    "closed_empty": "A_n closed\nInt(A_n)=empty",
    "dense_open": "U_n = X - A_n\nopen dense",
    "choose": "choose V_n\nclosure(V_n) avoids A_n",
    "shrink": "diam closure(V_n) -> 0",
    "complete": "complete metric\nor compact Hausdorff",
    "intersection": "nested closed sets\nhave a point x",
    "avoid_union": "x avoids every A_n",
    "baire": "union A_n has\nempty interior",
    "gdelta": "intersection U_n\nis dense",
}
proof_graph.add_nodes_from(proof_nodes)
proof_graph.add_edges_from([
    ("closed_empty", "dense_open"), ("closed_empty", "choose"), ("dense_open", "choose"),
    ("choose", "shrink"), ("shrink", "intersection"), ("complete", "intersection"),
    ("intersection", "avoid_union"), ("avoid_union", "baire"), ("dense_open", "gdelta"),
    ("baire", "gdelta"),
])
nx.set_node_attributes(proof_graph, {
    "closed_empty": 0, "dense_open": 0, "choose": 1, "shrink": 2, "complete": 2,
    "intersection": 3, "avoid_union": 4, "baire": 5, "gdelta": 5,
}, "layer")
pos = nx.multipartite_layout(proof_graph, subset_key="layer", align="vertical")
fig, ax = plt.subplots(figsize=(11, 4.8))
colors = ["#F6C85F" if n in {"closed_empty", "dense_open"} else "#9DD9A8" if n == "complete" else "#7EB6D9" for n in proof_graph.nodes]
nx.draw_networkx_edges(proof_graph, pos, ax=ax, arrows=True, arrowstyle="-|>", arrowsize=16, width=1.6, edge_color="#5A6472")
nx.draw_networkx_nodes(proof_graph, pos, ax=ax, node_size=3100, node_color=colors, edgecolors="#233142", linewidths=1.2)
nx.draw_networkx_labels(proof_graph, pos, labels=proof_nodes, ax=ax, font_size=8)
ax.set_title("Baire theorem proof state: what each hypothesis feeds")
ax.axis("off")
proof_graph_path = save_figure(fig, "baire-proof-state-graph.png")

forbidden = [0.5, 0.25, 0.75, 1 / 3, 2 / 3, 0.125, 0.875]
intervals = [(0.0, 1.0)]
round_rows = []
a, b = intervals[0]
for round_index, point in enumerate(forbidden, start=1):
    if a < point < b:
        left, right = (a, point), (point, b)
        chosen = right if (right[1] - right[0]) >= (left[1] - left[0]) else left
    else:
        chosen = (a, b)
    c, d = chosen
    margin = 0.18 * (d - c)
    a, b = c + margin, d - margin
    intervals.append((a, b))
    round_rows.append({
        "round": round_index, "obstacle": float(point), "left": float(a), "right": float(b),
        "length": float(b - a), "closure_avoids_obstacle": bool(b < point or a > point),
    })

fig, ax = plt.subplots(figsize=(11, 5.3))
for i, (left, right) in enumerate(intervals):
    y = len(intervals) - i
    ax.plot([left, right], [y, y], color="#2563A5", lw=8, solid_capstyle="round")
    ax.text(right + 0.018, y, f"V_{i}", va="center", fontsize=8)
for i, point in enumerate(forbidden, start=1):
    y = len(intervals) - i + 0.5
    ax.scatter([point], [y], color="#B23A48", s=42, zorder=5)
    ax.vlines(point, y - 0.32, y + 0.32, colors="#B23A48", lw=1.1)
    ax.text(point, y + 0.36, f"A_{i}", ha="center", fontsize=8, color="#7A2530")
limit_point = sum(intervals[-1]) / 2
ax.scatter([limit_point], [0.65], color="#1B7F5A", s=80, zorder=6)
ax.text(limit_point, 0.28, "surviving\npoint", ha="center", fontsize=8, color="#1B7F5A")
ax.set_xlim(-0.05, 1.05)
ax.set_ylim(0, len(intervals) + 1)
ax.set_yticks([])
ax.set_xlabel("unit interval model")
ax.set_title("Finite Baire category game: shrink while avoiding the next nowhere-dense obstacle")
ax.grid(axis="x", color="#D9DEE7", lw=0.6)
category_game_path = save_figure(fig, "baire-category-game.png")

def rationals_in_unit(max_den=8):
    values, seen = [], set()
    for denominator in range(1, max_den + 1):
        for numerator in range(0, denominator + 1):
            q = Fraction(numerator, denominator)
            if q not in seen:
                seen.add(q)
                values.append(q)
    return sorted(values, key=lambda q: (q.denominator, q.numerator))

removed = rationals_in_unit(9)[:28]
removed_float = np.array([float(q) for q in removed])
queries = [(0.02, 0.16), (0.18, 0.31), (0.43, 0.56), (0.62, 0.72), (0.83, 0.96)]
witness_rows = []
for left, right in queries:
    candidates = np.linspace(left, right, 700)
    distances = np.min(np.abs(candidates[:, None] - removed_float[None, :]), axis=1)
    idx = int(np.argmax(distances))
    witness_rows.append({
        "interval_left": left, "interval_right": right, "finite_witness": float(candidates[idx]),
        "distance_to_removed_sample": float(distances[idx]),
    })

fig, axes = plt.subplots(2, 1, figsize=(11, 4.8), sharex=True, gridspec_kw={"height_ratios": [1.0, 1.2]})
ax = axes[0]
ax.hlines(0, 0, 1, color="#233142", lw=2)
ax.scatter(removed_float, np.zeros_like(removed_float), color="#B23A48", s=28, zorder=4)
for q, x0 in zip(removed[:10], removed_float[:10]):
    ax.text(x0, 0.08, str(q), ha="center", rotation=90, fontsize=7, color="#7A2530")
ax.set_ylim(-0.25, 0.58)
ax.set_yticks([])
ax.set_title("Meagre sample: countably many closed singleton obstacles")
ax = axes[1]
ax.hlines(0, 0, 1, color="#233142", lw=2)
for row in witness_rows:
    left, right, witness = row["interval_left"], row["interval_right"], row["finite_witness"]
    ax.add_patch(Rectangle((left, -0.12), right - left, 0.24, color="#9DD9A8", alpha=0.42))
    ax.scatter([witness], [0], color="#1B7F5A", s=44, zorder=5)
    ax.text(witness, 0.16, "witness", ha="center", fontsize=7, color="#1B7F5A")
ax.scatter(removed_float, np.zeros_like(removed_float), color="#B23A48", s=15, alpha=0.55)
ax.set_ylim(-0.25, 0.48)
ax.set_yticks([])
ax.set_xlabel("sampled intervals in [0, 1]")
ax.set_title("Dense-open inspection: every test interval still has a point after finite removals")
fig.tight_layout()
dense_gdelta_path = save_figure(fig, "meagre-vs-dense.png")

baire_checks = {
    "proof_graph_nodes": proof_graph.number_of_nodes(),
    "proof_graph_edges": proof_graph.number_of_edges(),
    "category_game": {
        "rounds": round_rows,
        "final_interval": [float(intervals[-1][0]), float(intervals[-1][1])],
        "final_limit_point": float(limit_point),
        "lengths_strictly_decrease": bool(all(round_rows[i]["length"] < (round_rows[i - 1]["length"] if i else 1.0) for i in range(len(round_rows)))),
        "all_rounds_avoid_obstacle": bool(all(row["closure_avoids_obstacle"] for row in round_rows)),
    },
    "dense_gdelta_sample": {
        "removed_rational_count": len(removed),
        "test_intervals": witness_rows,
        "all_test_intervals_have_witness": bool(all(row["interval_left"] < row["finite_witness"] < row["interval_right"] for row in witness_rows)),
    },
}
save_json(baire_checks, CHAPTER_ARTIFACT, "checks", "baire-category-invariants.json")
save_table(round_rows, CHAPTER_ARTIFACT, "tables", "category-game-rounds.csv")
save_table(witness_rows, CHAPTER_ARTIFACT, "tables", "dense-gdelta-witnesses.csv")

for path in [proof_graph_path, category_game_path, dense_gdelta_path]:
    display_artifact(path, width=760)
baire_checks

## Dense `G_delta` Sets and Continuity Points

The dense-open formulation is the application engine. If each `U_n` is open and dense in a Baire space, then `intersection U_n` is dense. A typical residual set is therefore a `G_delta`: a countable intersection of open sets.

This perspective explains two source-span examples. The irrationals are the intersection of the dense open sets `R - {q_n}`, where the `q_n` enumerate the rationals. The finite picture above removes only the first few rational points, but the inspection target is correct: every open interval should keep witnesses after each finite stage, and Baire lets the countable intersection stay dense.

For pointwise limits, the source builds closed sets `A_N(epsilon)` where a tail of continuous functions is uniformly Cauchy at a point. Baire forces one of these closed sets to have interior inside any test open set, producing dense open continuity control. The proof graph records that dependency rather than hiding it in a long paragraph.

## Nowhere-Differentiable Functions: Dense Roughness

Section 49 applies Baire in the complete metric space `C([0,1], R)` with the uniform metric. For each positive integer `n`, define a set `U_n` of continuous functions whose small difference quotients are large everywhere. The source proof checks three facts: functions in the intersection of all `U_n` are nowhere differentiable, each `U_n` is open, and each `U_n` is dense.

The dense part is the constructive-looking part: start with a continuous function, approximate it by a piecewise-linear graph, then replace flat pieces by tiny sawteeth whose slopes are very large. Uniform error stays small because the teeth have small amplitude; difference quotients get large because the teeth have high frequency.

The next two artifacts are not a proof of nowhere differentiability. They are diagnostics for the proof mechanism: small vertical error can coexist with large local slopes, and finite samples can verify the intended `U_n` threshold for several `n`.

In [ ]:
x = np.linspace(0, 1, 2400)
base = base_curve(x)
fig, axes = plt.subplots(2, 1, figsize=(11, 7.2), sharex=True, gridspec_kw={"height_ratios": [1.8, 1.0]})
ax = axes[0]
ax.plot(x, base, color="#233142", lw=2.0, label="smooth base h")
for levels, color in zip([2, 4, 7], ["#2D9CDB", "#F2994A", "#9B51E0"]):
    ax.plot(x, rough_series(x, levels=levels), color=color, lw=1.35, label=f"h plus {levels} sawtooth levels")
ax.set_title("Uniformly small sawtooth perturbations can make slopes large")
ax.set_ylabel("function value")
ax.legend(loc="upper right", ncol=2, fontsize=8)
ax.grid(color="#D9DEE7", lw=0.6)
ax = axes[1]
for k in range(7):
    amp, freq = 0.055 * (0.62 ** k), 2 ** (k + 1)
    ax.bar(k + 1, amp, color="#7EB6D9")
    ax.text(k + 1, amp + 0.002, f"freq {freq}", ha="center", fontsize=7, rotation=90)
ax.plot(np.arange(1, 8), [0.055 * (0.62 ** k) * 2 ** (k + 2) for k in range(7)], color="#B23A48", marker="o", label="sawtooth slope scale")
ax.set_xlabel("perturbation level")
ax.set_ylabel("amplitude / slope scale")
ax.set_title("Amplitudes shrink while slope scale grows")
ax.legend(fontsize=8)
ax.grid(axis="y", color="#D9DEE7", lw=0.6)
fig.tight_layout()
rough_function_path = save_figure(fig, "nowhere-differentiable-approximation.png")

sample_grid = np.linspace(0, 1, 801)
diagnostic_rows = [finite_secant_score(n, sample_grid) for n in range(1, 7)]
quotient_df = pd.DataFrame(diagnostic_rows)
quotient_df.to_csv(TABLES / "difference-quotient-diagnostics.csv", index=False)

fig, ax = plt.subplots(figsize=(10.5, 4.8))
ax.plot(quotient_df["n"], quotient_df["min_sampled_witness"], marker="o", color="#1B7F5A", lw=2, label="minimum sampled witness quotient")
ax.plot(quotient_df["n"], quotient_df["median_sampled_witness"], marker="s", color="#2563A5", lw=2, label="median sampled witness quotient")
ax.plot(quotient_df["n"], quotient_df["n"], color="#B23A48", ls="--", lw=1.6, label="threshold n")
for row in diagnostic_rows:
    ax.text(row["n"], row["min_sampled_witness"] + 0.25, f"f={row['frequency']}", ha="center", fontsize=7)
ax.set_xlabel("n in the finite U_n diagnostic")
ax.set_ylabel("absolute difference quotient")
ax.set_title("Finite diagnostic for dense U_n intuition: every sampled x sees a quotient above n")
ax.grid(color="#D9DEE7", lw=0.6)
ax.legend(fontsize=8)
diffquot_path = save_figure(fig, "difference-quotient-diagnostics.png")

rough_checks = {
    "rough_series_levels": 7,
    "amplitude_ratio": 0.62,
    "tail_bound_after_7_levels": float(0.055 * (0.62 ** 7) / (1 - 0.62)),
    "diagnostics": diagnostic_rows,
    "all_min_witnesses_exceed_n": bool(all(row["min_sampled_witness"] > row["n"] for row in diagnostic_rows)),
    "all_uniform_errors_below_epsilon": bool(all(row["uniform_error_bound"] < row["epsilon"] for row in diagnostic_rows)),
}
save_json(rough_checks, CHAPTER_ARTIFACT, "checks", "rough-function-diagnostics.json")

for path in [rough_function_path, diffquot_path]:
    display_artifact(path, width=760)
quotient_df

## Covering Dimension: Refinement Order

Covering dimension is not the number of points in a finite model. It is an overlap guarantee for arbitrary open covers.

A collection has order `m + 1` when some point lies in `m + 1` sets and no point lies in more than `m + 1` sets. A space has `dim X <= m` if every open cover has an open refinement with order at most `m + 1`.

For compact metric spaces, the Lebesgue number idea lets a sufficiently fine mesh refine a given cover. The diagrams below use open-star covers: in a subdivided interval, a point is controlled by at most two neighboring vertex stars; in a triangulated square, a point is controlled by at most three vertex stars. This is the visible version of the bound `m + 1`.

In [ ]:
coarse_intervals = [(-0.08, 0.42), (0.20, 0.72), (0.58, 1.08)]
star_intervals = [(-0.08, 0.26), (0.18, 0.52), (0.44, 0.78), (0.70, 1.08)]
coarse_order, _, coarse_counts = max_interval_order(coarse_intervals)
star_order, _, star_counts = max_interval_order(star_intervals)

mesh_n = 5
vertices = np.array([(i / mesh_n, j / mesh_n) for j in range(mesh_n + 1) for i in range(mesh_n + 1)])
def vid(i, j):
    return j * (mesh_n + 1) + i
triangles = []
for j in range(mesh_n):
    for i in range(mesh_n):
        triangles.append((vid(i, j), vid(i + 1, j), vid(i + 1, j + 1)))
        triangles.append((vid(i, j), vid(i + 1, j + 1), vid(i, j + 1)))
triangles = np.array(triangles, dtype=int)
sample_points, orders = [], []
for px in np.linspace(0.01, 0.99, 70):
    for py in np.linspace(0.01, 0.99, 70):
        i, j = min(int(px * mesh_n), mesh_n - 1), min(int(py * mesh_n), mesh_n - 1)
        local_x, local_y = px * mesh_n - i, py * mesh_n - j
        tri = (vid(i, j), vid(i + 1, j), vid(i + 1, j + 1)) if local_y <= local_x else (vid(i, j), vid(i + 1, j + 1), vid(i, j + 1))
        sample_points.append((px, py))
        orders.append(len(set(tri)))
sample_points, orders = np.asarray(sample_points), np.asarray(orders)
mesh_order = int(orders.max())

fig = plt.figure(figsize=(12, 7.4))
gs = fig.add_gridspec(2, 2, height_ratios=[1, 1.1])
ax = fig.add_subplot(gs[0, 0])
for left, right in coarse_intervals:
    ax.add_patch(Rectangle((left, 0.25), right - left, 0.18, color="#F6C85F", alpha=0.55))
for left, right in star_intervals:
    ax.add_patch(Rectangle((left, -0.05), right - left, 0.18, color="#7EB6D9", alpha=0.65))
for v in np.linspace(0, 1, 4):
    ax.scatter([v], [-0.18], color="#233142", s=25)
ax.hlines([0.34, 0.04], 0, 1, colors="#233142", lw=1.2)
ax.text(1.02, 0.34, f"coarse order {coarse_order}", va="center", fontsize=8)
ax.text(1.02, 0.04, f"refined order {star_order}", va="center", fontsize=8)
ax.set_xlim(-0.12, 1.18)
ax.set_ylim(-0.32, 0.6)
ax.set_yticks([])
ax.set_title("Interval: open-star refinement has order <= 2")
ax.set_xlabel("[0, 1]")

ax = fig.add_subplot(gs[0, 1])
for tri in triangles:
    ax.add_patch(Polygon(vertices[list(tri)], closed=True, facecolor="none", edgecolor="#59677A", lw=0.7))
sc = ax.scatter(sample_points[:, 0], sample_points[:, 1], c=orders, cmap="viridis", s=8, vmin=1, vmax=3, alpha=0.75)
ax.set_aspect("equal")
ax.set_xlim(-0.03, 1.03)
ax.set_ylim(-0.03, 1.03)
ax.set_title("Square triangulation: sampled open-star order <= 3")
ax.set_xticks([])
ax.set_yticks([])
cbar = fig.colorbar(sc, ax=ax, fraction=0.046, pad=0.03)
cbar.set_label("sets containing point")

ax = fig.add_subplot(gs[1, :])
ax.axis("off")
steps = [(0.05, "arbitrary\nopen cover"), (0.28, "Lebesgue\nscale"), (0.51, "mesh or\ntriangulation"), (0.74, "open-star\nrefinement"), (0.94, "order\n<= m+1")]
for x0, label in steps:
    ax.add_patch(Rectangle((x0 - 0.075, 0.35), 0.15, 0.28, facecolor="#F3F6FA", edgecolor="#233142", lw=1.1))
    ax.text(x0, 0.49, label, ha="center", va="center", fontsize=10)
for (x0, _), (x1, _) in zip(steps, steps[1:]):
    ax.add_patch(FancyArrowPatch((x0 + 0.08, 0.49), (x1 - 0.08, 0.49), arrowstyle="-|>", mutation_scale=14, color="#2563A5", lw=1.5))
ax.text(0.5, 0.18, "The diagram is a finite witness for the refinement-order invariant, not a replacement for the universal quantifier over covers.", ha="center", fontsize=9, color="#4A5568")
fig.suptitle("Covering dimension as controlled overlap after refinement", y=0.99)
fig.tight_layout()
dimension_refinement_path = save_figure(fig, "dimension-refinement-covers.png")

t_values = np.array([-1.0, -0.45, 0.0, 0.55, 1.05])
points3d = np.c_[t_values, t_values ** 2, t_values ** 3]
points3d = (points3d - points3d.mean(axis=0)) / points3d.std(axis=0)
edges = list(itertools.combinations(range(len(points3d)), 2))
fig = plt.figure(figsize=(8.2, 6.8))
ax = fig.add_subplot(111, projection="3d")
ax.add_collection3d(Line3DCollection([(points3d[i], points3d[j]) for i, j in edges], colors="#2563A5", linewidths=1.1, alpha=0.82))
ax.scatter(points3d[:, 0], points3d[:, 1], points3d[:, 2], color="#B23A48", s=58, depthshade=True)
for idx, point in enumerate(points3d):
    ax.text(point[0], point[1], point[2], f"v{idx+1}", fontsize=8)
ax.set_title("General-position graph model in R^3: moment-curve vertices")
ax.set_xlabel("x")
ax.set_ylabel("y")
ax.set_zlabel("z")
ax.view_init(elev=20, azim=-55)
general_position_path = save_figure(fig, "general-position-embedding.png")

four_tuple_rows = []
for combo in itertools.combinations(range(len(points3d)), 4):
    det = determinant_for_points(points3d[list(combo)])
    four_tuple_rows.append({"vertices": "-".join(str(i + 1) for i in combo), "affine_3_volume_det": det, "abs_det": abs(det)})
three_tuple_rows = []
for combo in itertools.combinations(range(len(points3d)), 3):
    pts = points3d[list(combo)]
    three_tuple_rows.append({"vertices": "-".join(str(i + 1) for i in combo), "affine_rank": int(np.linalg.matrix_rank(pts[1:] - pts[0], tol=1e-10))})
save_table(four_tuple_rows, CHAPTER_ARTIFACT, "tables", "general-position-determinants.csv")
save_table(three_tuple_rows, CHAPTER_ARTIFACT, "tables", "general-position-ranks.csv")

fig, axes = plt.subplots(1, 2, figsize=(11, 4.8))
ax = axes[0]
theta = np.linspace(0, 2 * np.pi, 400)
ax.plot(np.cos(theta), np.sin(theta), color="#233142", lw=2)
for start, stop, color in [(0.05, 1.45, "#7EB6D9"), (1.25, 2.85, "#F6C85F"), (2.65, 4.25, "#9DD9A8"), (4.05, 5.85, "#F2994A")]:
    th = np.linspace(start, stop, 100)
    ax.plot(1.04 * np.cos(th), 1.04 * np.sin(th), color=color, lw=7, solid_capstyle="round")
ax.text(0, -1.32, "compact 1-manifold: finite arc charts", ha="center", fontsize=9)
ax.set_aspect("equal")
ax.axis("off")
ax.set_title("Local Euclidean patches")
ax = axes[1]
for idx, (cx, cy) in enumerate([(0.25, 0.34), (0.46, 0.55), (0.66, 0.36), (0.50, 0.25), (0.78, 0.62)]):
    ax.add_patch(Circle((cx, cy), 0.23, facecolor=["#7EB6D9", "#F6C85F", "#9DD9A8", "#F2994A", "#B6A6E9"][idx], edgecolor="#233142", alpha=0.45, lw=1.1))
    ax.text(cx, cy, f"U{idx+1}", ha="center", va="center", fontsize=9)
ax.add_patch(FancyArrowPatch((0.18, 0.88), (0.82, 0.88), arrowstyle="-|>", mutation_scale=14, color="#2563A5", lw=1.4))
ax.text(0.5, 0.94, "finite union of chart domains", ha="center", fontsize=9)
ax.text(0.5, 0.78, "dim <= m by closed/finite-union machinery\nthen compact m-manifold embeds in R^(2m+1)", ha="center", fontsize=9)
ax.set_xlim(0, 1)
ax.set_ylim(0, 1.02)
ax.set_aspect("equal")
ax.axis("off")
ax.set_title("Atlas to dimension bound")
fig.tight_layout()
atlas_path = save_figure(fig, "locally-euclidean-atlas.png")

cover_rows = [
    {"model": "interval_coarse", "dimension_m": 1, "max_order": coarse_order, "expected_bound": 2, "coverage_sampled": bool(np.all(coarse_counts > 0))},
    {"model": "interval_open_star_refinement", "dimension_m": 1, "max_order": star_order, "expected_bound": 2, "coverage_sampled": bool(np.all(star_counts > 0))},
    {"model": "square_triangulated_open_star", "dimension_m": 2, "max_order": mesh_order, "expected_bound": 3, "coverage_sampled": True},
]
save_table(cover_rows, CHAPTER_ARTIFACT, "tables", "cover-refinement-order.csv")
dimension_checks = {
    "cover_rows": cover_rows,
    "interval_refinement_order_within_bound": bool(star_order <= 2),
    "triangulated_square_order_within_bound": bool(mesh_order <= 3),
    "moment_curve_general_position": {
        "min_abs_four_point_det": float(min(row["abs_det"] for row in four_tuple_rows)),
        "all_four_tuples_affinely_independent": bool(all(row["abs_det"] > 1e-8 for row in four_tuple_rows)),
        "all_three_tuples_noncollinear": bool(all(row["affine_rank"] == 2 for row in three_tuple_rows)),
        "embedding_dimension_for_m_equals_1": 2 * 1 + 1,
    },
}
save_json(dimension_checks, CHAPTER_ARTIFACT, "checks", "dimension-refinement-checks.json")

for path in [dimension_refinement_path, general_position_path, atlas_path]:
    display_artifact(path, width=760)
pd.DataFrame(cover_rows)

## Applied Lab: Vary the Finite Shadows

The source theorems are infinite statements, but the proof moves have finite shadows. The HTML artifact below lets you vary two finite parameters.

The first panel shows how many Baire-game rounds have been played. More rounds mean more obstacles have been avoided, while the surviving interval shrinks.

The second panel changes mesh scale in a square. The open-star refinement keeps the order bound at `3`, matching the `m + 1` pattern for a two-dimensional model, while making the sets smaller.

In [ ]:
fig = make_subplots(
    rows=1, cols=2,
    subplot_titles=("Finite Baire game", "Triangulated cover order"),
    specs=[[{"type": "xy"}, {"type": "xy"}]],
)
for r in range(1, len(intervals)):
    left, right = intervals[r]
    y = len(intervals) - r
    fig.add_trace(go.Scatter(x=[left, right], y=[y, y], mode="lines", line=dict(color="#2563A5", width=10), name=f"V_{r}", visible=(r <= 3)), row=1, col=1)
    point = forbidden[r - 1]
    fig.add_trace(go.Scatter(x=[point], y=[y + 0.32], mode="markers+text", marker=dict(color="#B23A48", size=9), text=[f"A{r}"], textposition="top center", showlegend=False, visible=(r <= 3)), row=1, col=1)
fig.add_trace(go.Scatter(x=[limit_point], y=[0.35], mode="markers+text", marker=dict(color="#1B7F5A", size=12), text=["survivor"], textposition="bottom center", name="surviving point", visible=True), row=1, col=1)

mesh_trace_indices = []
for n in [2, 3, 4, 5]:
    xs2 = np.linspace(0, 1, n + 1)
    sx, sy = [], []
    for v in xs2:
        sx += [0, 1, None, v, v, None]
        sy += [v, v, None, 0, 1, None]
    idx = len(fig.data)
    mesh_trace_indices.append(idx)
    fig.add_trace(go.Scatter(x=sx, y=sy, mode="lines", line=dict(color="#59677A", width=1), name=f"mesh {n}x{n}", visible=(n == 3)), row=1, col=2)
    cx, cy = [], []
    for i in range(n):
        for j in range(n):
            cx.append((i + 0.5) / n)
            cy.append((j + 0.5) / n)
    fig.add_trace(go.Scatter(x=cx, y=cy, mode="markers", marker=dict(size=10, color="#9DD9A8"), text=["order <= 3"] * len(cx), hovertemplate="%{text}<extra></extra>", showlegend=False, visible=(n == 3)), row=1, col=2)

steps = []
survivor_index = 2 * (len(intervals) - 1)
for r in range(1, len(intervals)):
    visible = [False] * len(fig.data)
    for i in range(2 * r):
        visible[i] = True
    visible[survivor_index] = True
    for idx, n in zip(mesh_trace_indices, [2, 3, 4, 5]):
        visible[idx] = (n == 3)
        visible[idx + 1] = (n == 3)
    steps.append(dict(method="update", args=[{"visible": visible}], label=str(r)))

mesh_steps = []
for idx, n in zip(mesh_trace_indices, [2, 3, 4, 5]):
    visible = [False] * len(fig.data)
    for i in range(6):
        visible[i] = True
    visible[survivor_index] = True
    visible[idx] = True
    visible[idx + 1] = True
    mesh_steps.append(dict(method="update", args=[{"visible": visible}], label=f"{n}x{n}"))

fig.update_xaxes(range=[-0.05, 1.05], row=1, col=1, title_text="unit interval")
fig.update_yaxes(range=[0, len(intervals)], showticklabels=False, row=1, col=1)
fig.update_xaxes(range=[-0.03, 1.03], row=1, col=2, scaleanchor="y2", scaleratio=1, title_text="square model")
fig.update_yaxes(range=[-0.03, 1.03], showticklabels=False, row=1, col=2)
fig.update_layout(
    title="Chapter 8 lab: finite shadows of Baire category and covering dimension",
    width=1050, height=540, margin=dict(l=40, r=30, t=80, b=90),
    sliders=[
        dict(active=2, currentvalue={"prefix": "Baire rounds: "}, pad={"t": 42}, steps=steps, x=0.08, len=0.38),
        dict(active=1, currentvalue={"prefix": "Mesh scale: "}, pad={"t": 42}, steps=mesh_steps, x=0.58, len=0.34),
    ],
)
lab_html = "\n".join(line.rstrip() for line in fig.to_html(include_plotlyjs=True, full_html=True).splitlines())
lab_path = save_html(lab_html, CHAPTER_ARTIFACT, "html", "baire-dimension-lab.html")
display_artifact(lab_path, width="100%", height=560)
lab_path

## Sanity Checks

The final cell checks both artifact integrity and chapter-specific invariants. The assertions intentionally mix visual checks with mathematical shadows: if an artifact is missing, blank, or stale, the notebook should fail; if the finite diagnostics no longer match the theorem intuition, the notebook should fail too.

In [ ]:
required_artifacts = [
    FIGURES / "baire-proof-state-graph.png",
    FIGURES / "baire-category-game.png",
    FIGURES / "meagre-vs-dense.png",
    FIGURES / "nowhere-differentiable-approximation.png",
    FIGURES / "difference-quotient-diagnostics.png",
    FIGURES / "dimension-refinement-covers.png",
    FIGURES / "general-position-embedding.png",
    FIGURES / "locally-euclidean-atlas.png",
    HTML / "baire-dimension-lab.html",
    CHECKS / "baire-category-invariants.json",
    CHECKS / "rough-function-diagnostics.json",
    CHECKS / "dimension-refinement-checks.json",
    TABLES / "category-game-rounds.csv",
    TABLES / "dense-gdelta-witnesses.csv",
    TABLES / "difference-quotient-diagnostics.csv",
    TABLES / "cover-refinement-order.csv",
    TABLES / "general-position-determinants.csv",
]
assert_artifacts(required_artifacts, min_size=64)
png_stats = {path.name: assert_png_nonblank(path) for path in required_artifacts if path.suffix == ".png"}
baire_data = json.loads((CHECKS / "baire-category-invariants.json").read_text(encoding="utf-8"))
rough_data = json.loads((CHECKS / "rough-function-diagnostics.json").read_text(encoding="utf-8"))
dimension_data = json.loads((CHECKS / "dimension-refinement-checks.json").read_text(encoding="utf-8"))

assert baire_data["category_game"]["all_rounds_avoid_obstacle"]
assert baire_data["category_game"]["lengths_strictly_decrease"]
assert baire_data["dense_gdelta_sample"]["all_test_intervals_have_witness"]
assert rough_data["all_min_witnesses_exceed_n"]
assert rough_data["all_uniform_errors_below_epsilon"]
assert dimension_data["interval_refinement_order_within_bound"]
assert dimension_data["triangulated_square_order_within_bound"]
assert dimension_data["moment_curve_general_position"]["all_four_tuples_affinely_independent"]
assert dimension_data["moment_curve_general_position"]["all_three_tuples_noncollinear"]

chapter_summary = {
    "chapter": 8,
    "source_span": "printed pages 292-316; sections 48-50",
    "core_identities_checked": {
        "finite_category_game_avoids_each_obstacle": baire_data["category_game"]["all_rounds_avoid_obstacle"],
        "dense_gdelta_sample_witnesses": baire_data["dense_gdelta_sample"]["all_test_intervals_have_witness"],
        "finite_U_n_secants_exceed_thresholds": rough_data["all_min_witnesses_exceed_n"],
        "cover_order_bounds": [row for row in dimension_data["cover_rows"]],
        "general_position_for_menger_nobeling_shadow": dimension_data["moment_curve_general_position"],
    },
    "artifacts_checked": [book_relative(path) for path in required_artifacts],
    "png_count": len(png_stats),
}
save_json(chapter_summary, CHAPTER_ARTIFACT, "checks", "chapter-summary.json")
print("Checked", len(required_artifacts), "artifacts and", len(png_stats), "PNG visuals.")
print("Representative PNG stats:", image_stats(FIGURES / "dimension-refinement-covers.png"))
chapter_summary

## Takeaways

- A Baire space is one where countably many closed sets with empty interior cannot fill a nonempty open set.
- The dense-open form is the application engine: countable intersections of dense open sets remain dense.
- The nowhere-differentiable theorem uses completeness of `C([0,1], R)` and dense open `U_n` conditions, not an explicit formula alone.
- Covering dimension is measured by refinement order: `dim X <= m` means every open cover can be refined so that at most `m + 1` refined sets meet at any point.
- The `2m + 1` embedding theorem is driven by Baire category plus general position: after refining covers to order `m + 1`, affine independence in `R^{2m+1}` separates points.